<div style="text-align: center;">
    <h1 style="color: #FF6347;">Self-Guided Lab: Retrieval-Augmented Generation (RAGs)</h1>
</div>

<div style="text-align: center;">
    <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZ3FsdzRveTBrenMxM3VnbDMwaTJxN2NnZm50aGFibXk1NzNnY2Q0MCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/LR5ZBwZHv02lmpVoEU/giphy.gif" alt="NLP Gif" style="width: 300px; height: 150px; object-fit: cover; object-position: center;">
</div>

<h1 style="color: #FF6347;">Data Storage & Retrieval</h1>


<h2 style="color: #FF8C00;">PyPDFLoader</h2>

`PyPDFLoader` is a lightweight Python library designed to streamline the process of loading and parsing PDF documents for text processing tasks. It is particularly useful in Retrieval-Augmented Generation workflows where text extraction from PDFs is required.

- **What Does PyPDFLoader Do?**
  - Extracts text from PDF files, retaining formatting and layout.
  - Simplifies the preprocessing of document-based datasets.
  - Supports efficient and scalable loading of large PDF collections.

- **Key Features:**
  - Compatible with popular NLP libraries and frameworks.
  - Handles multi-page PDFs and embedded images (e.g., OCR-compatible setups).
  - Provides flexible configurations for structured text extraction.

- **Use Cases:**
  - Preparing PDF documents for retrieval-based systems in RAGs.
  - Automating the text extraction pipeline for document analysis.
  - Creating datasets from academic papers, technical manuals, and reports.


In [ ]:
%pip install langchain langchain_community pypdf
%pip install termcolor langchain_openai langchain-huggingface sentence-transformers chromadb langchain_chroma tiktoken openai python-dotenv


In [ ]:
import os
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
import warnings
warnings.filterwarnings('ignore')


In [4]:
import os
import warnings
warnings.filterwarnings("ignore")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from dotenv import load_dotenv
from termcolor import colored
import openai


<h3 style="color: #FF8C00;">Loading the Documents</h3>

In [11]:
# File path for the document

file_path = "../LAB/ai-for-everyone.pdf"

<h3 style="color: #FF8C00;">Documents into pages</h3>

The `PyPDFLoader` library allows efficient loading and splitting of PDF documents into smaller, manageable parts for NLP tasks.

This functionality is particularly useful in workflows requiring granular text processing, such as Retrieval-Augmented Generation (RAG).


In [12]:
# Load and split the document
loader = PyPDFLoader(file_path)
pages = loader.load_and_split()
len(pages)

297

<h3 style="color: #FF8C00;">Pages into Chunks</h3>


####  RecursiveCharacterTextSplitter in LangChain

The `RecursiveCharacterTextSplitter` is the **recommended splitter** in LangChain when you want to break down long documents into smaller, semantically meaningful chunks — especially useful in **RAG pipelines**, where clean context chunks lead to better LLM responses.

####  Parameters

| Parameter       | Description                                                                 |
|-----------------|-----------------------------------------------------------------------------|
| `chunk_size`    | The **maximum number of characters** allowed in a chunk (e.g., `1000`).     |
| `chunk_overlap` | The number of **overlapping characters** between consecutive chunks (e.g., `200`). This helps preserve context continuity. |

####  How it works
`RecursiveCharacterTextSplitter` attempts to split the text **intelligently**, trying the following separators in order:
1. Paragraphs (`"\n\n"`)
2. Lines (`"\n"`)
3. Sentences or words (`" "`)
4. Individual characters (as a last resort)

This makes it ideal for handling **natural language documents**, such as PDFs, articles, or long reports, without breaking sentences or paragraphs in awkward ways.



In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(pages)

len(chunks)

1096

####  Alternative: CharacterTextSplitter

`CharacterTextSplitter` is a simpler splitter that breaks text into chunks based **purely on character count**, without trying to preserve any natural language structure.

##### Example:
```python
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
````

This method is faster and more predictable but may split text in the middle of a sentence or paragraph, which can hurt performance in downstream tasks like retrieval or QA.

---

#### Comparison Table

| Feature                        | RecursiveCharacterTextSplitter | CharacterTextSplitter     |
| ------------------------------ | ------------------------------ | ------------------------- |
| Structure-aware splitting      |  Yes                          |  No                      |
| Preserves sentence/paragraphs  |  Yes                          |  No                      |
| Risk of splitting mid-sentence |  Minimal                     |  High                   |
| Ideal for RAG/document QA      |  Highly recommended           |  Only if structured text |
| Performance speed              |  Slightly slower             |  Faster                  |

---

#### Recommendation

Use `RecursiveCharacterTextSplitter` for most real-world document processing tasks, especially when building RAG pipelines or working with structured natural language content like PDFs or articles.

## Best Practices for Choosing Chunk Size in RAG

### Best Practices for Chunk Size in RAG

| Factor                      | Recommendation                                                                                                                                                                                          |
| ---------------------------| ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **LLM context limit**       | Choose a chunk size that lets you retrieve multiple chunks **without exceeding the model’s token limit**. For example, GPT-4o supports 128k tokens, but with GPT-3.5 (16k) or GPT-4 (32k), keep it modest. |
| **Chunk size (in characters)** | Typically: **500–1,000 characters** per chunk → ~75–200 tokens. This fits well for retrieval + prompt without context overflow.                                                                           |
| **Chunk size (in tokens)**  | If using token-based splitter (e.g. `TokenTextSplitter`): aim for **100–300 tokens** per chunk.                                                                                                            |
| **Chunk overlap**           | Use **overlap of 10–30%** (e.g., 100–300 characters or ~50 tokens) to preserve context across chunk boundaries and avoid cutting off important ideas mid-sentence.                                        |
| **Document structure**      | Use **`RecursiveCharacterTextSplitter`** to preserve semantic boundaries (paragraphs, sentences) instead of arbitrary cuts.                                                                                |
| **Task type**               | For **question answering**, smaller chunks (~500–800 chars) reduce noise.<br>For **summarization**, slightly larger chunks (~1000–1500) are OK.                                                          |
| **Embedding model**         | Some models (e.g., `text-embedding-3-large`) can handle long input. But still, smaller chunks give **finer-grained retrieval**, which improves relevance.                                                  |
| **Query type**              | If users ask **very specific questions**, small focused chunks are better. For broader queries, bigger chunks might help.                                                                                  |


### Rule of Thumb

| Use Case                 | Chunk Size      | Overlap |
| ------------------------| --------------- | ------- |
| Factual Q&A              | 500–800 chars   | 100–200 |
| Summarization            | 1000–1500 chars | 200–300 |
| Technical documents      | 400–700 chars   | 100–200 |
| Long reports/books       | 800–1200 chars  | 200–300 |
| Small LLMs (≤16k tokens) | ≤800 chars      | 100–200 |


### Avoid

- Chunks >2000 characters: risks context overflow.
- No overlap: may lose key information between chunks.



<h2 style="color: #FF8C00;">Embeddings</h2>

Embeddings transform text into dense vector representations, capturing semantic meaning and contextual relationships. They are essential for efficient document retrieval and similarity analysis.

- **What are OpenAI Embeddings?**
  - Pre-trained embeddings like `text-embedding-3-large` generate high-quality vector representations for text.
  - Encapsulate semantic relationships in the text, enabling robust NLP applications.

- **Key Features of `text-embedding-3-large`:**
  - Large-scale embedding model optimized for accuracy and versatility.
  - Handles diverse NLP tasks, including retrieval, classification, and clustering.
  - Ideal for applications with high-performance requirements.

- **Benefits:**
  - Reduces the need for extensive custom training.
  - Provides state-of-the-art performance in retrieval-augmented systems.
  - Compatible with RAGs to create powerful context-aware models.


In [15]:
#from langchain.embeddings import OpenAIEmbeddings
#from dotenv import load_dotenv

In [16]:
load_dotenv()

True

In [ ]:
api_key = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

<h2 style="color: #FF8C00;">ChromaDB</h2>

ChromaDB is a versatile vector database designed for efficiently storing and retrieving embeddings. It integrates seamlessly with embedding models to enable high-performance similarity search and context-based retrieval.

### Workflow Overview:
- **Step 1:** Generate embeddings using a pre-trained model (e.g., OpenAI's `text-embedding-3-large`).
- **Step 2:** Store the embeddings in ChromaDB for efficient retrieval and similarity calculations.
- **Step 3:** Use the stored embeddings to perform searches, matching, or context-based retrieval.

### Key Features of ChromaDB:
- **Scalability:** Handles large-scale datasets with optimized indexing and search capabilities.
- **Speed:** Provides fast and accurate retrieval of embeddings for real-time applications.
- **Integration:** Supports integration with popular frameworks and libraries for embedding generation.

In [19]:
#from langchain.vectorstores import Chroma

In [20]:
db = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db_LAB")
print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


<h1 style="color: #FF6347;">Retrieving Documents</h1>


### Exercice1: Write a user question that someone might ask about your book’s topic or content.

In [ ]:
user_question = "What ethical dilemmas and data challenges are present in AI?" # User question
retrieved_docs = db.similarity_search(user_question, k=10) # k is the number of documents to retrieve

In [22]:
# Display top results
for i, doc in enumerate(retrieved_docs[:3]): # Display top 3 results
    print(f"Document {i+1}:\n{doc.page_content[36:1000]}") # Display content

Document 1:
will be needed to address the various challenges with regards to  
the development of artificial intelligence. Which formal method can be used 
to test for algorithmic bias? Can we identify simple to use measures to assess 
bias, similar to the way we assess reliability and validity? What is the best way to 
bridge (deep) learning and privacy? Should learning be conducted on the user 
side (with algorithms requiring new data)? Or should data be transferred to a 
trusted intermediary who performs the analysis on behalf of firms? Do users 
need to be compensated in one way or another for data or resources provided? 
Moreover, how can the refusal to share data lead to biases in the data available 
for learning? Which data sources can and should be used for algorithmic learn-
ing? Are there certain types of data that should be ‘off-limits’? What role will 
interdisciplinary AI teams play in establishing coexistence between humans
Document 2:
ous and problematic uses of the tech

<h2 style="color: #FF8C00;">Preparing Content for GenAI</h2>

In [23]:
def _get_document_prompt(docs):
    prompt = "\n"
    for doc in docs:
        prompt += "\nContent:\n"
        prompt += doc.page_content + "\n\n"
    return prompt

In [24]:
# Generate a formatted context from the retrieved documents
formatted_context = _get_document_prompt(retrieved_docs)
print("Context formatted for GPT model.")

Context formatted for GPT model.


<h2 style="color: #FF8C00;">ChatBot Architecture</h2>

### Exercice2: Write a prompt that is relevant and tailored to the content and style of your book.

In [30]:
prompt = f"""
## SYSTEM ROLE
You are a knowledgeable, objective, and critical AI researcher chatbot focused on **Artificial Intelligence ethics, governance, and societal impact**, grounded in **critical AI studies**.

You must base all responses **exclusively** on the provided academic book content.
Do not speculate or introduce external knowledge.

## USER QUESTION
"{user_question}"

## CONTEXT
Book analyzed:
**AI for Everyone? Critical Perspectives**, edited by Pieter Verdegem.

Relevant excerpts:
'''
{formatted_context}
'''

## GUIDELINES
- Use only the information in the `CONTEXT`.
- If the answer is not present, state:
  **"The provided context does not contain this information."**
- Reference the book title and page numbers for all claims.
- Emphasize ethical, social, and philosophical critiques of AI.
- Maintain an academic, concise, and neutral tone.
- Format the response in Markdown.

## TASK
When supported by the context:
1. Identify key critical perspectives on AI.
2. Discuss human–machine relationships.
3. Highlight ethical and social concerns.
4. Reference relevant authors or viewpoints.
5. Provide a critical assessment of AI’s role in society.

## RESPONSE FORMAT
# [Brief Title]

## Critical Perspectives on AI
[Clear, structured analysis based on the book.]

## Ethical and Social Implications
[Discussion grounded in the provided content.]

## Conclusion
[Critical synthesis derived from the text.]

**Source**:
• AI for Everyone? Critical Perspectives, Page(s): [...]
"""


In [31]:
print(prompt)


## SYSTEM ROLE
You are a knowledgeable, objective, and critical AI researcher chatbot focused on **Artificial Intelligence ethics, governance, and societal impact**, grounded in **critical AI studies**.

You must base all responses **exclusively** on the provided academic book content.
Do not speculate or introduce external knowledge.

## USER QUESTION
"What ethical dilemmas and data challenges are present in AI?"

## CONTEXT
Book analyzed:
**AI for Everyone? Critical Perspectives**, edited by Pieter Verdegem.

Relevant excerpts:
'''


Content:
30 AI for Everyone?
Future research will be needed to address the various challenges with regards to  
the development of artificial intelligence. Which formal method can be used 
to test for algorithmic bias? Can we identify simple to use measures to assess 
bias, similar to the way we assess reliability and validity? What is the best way to 
bridge (deep) learning and privacy? Should learning be conducted on the user 
side (with algorithms re

In [27]:
import openai

### Exercice3: Tune parameters like temperature, and penalties to control how creative, focused, or varied the model's responses are.

In [32]:

# Set up GPT client and parameters
client = openai.OpenAI()
model_params = {
    'model': 'gpt-4o-mini',
    'temperature': 0.7,  # Increase creativity
    'max_tokens': 4000,  # Allow for longer responses
    'top_p': 0.9,        # Use nucleus sampling
    'frequency_penalty': 0.5,  # Reduce repetition
    'presence_penalty': 0.6   # Encourage new topics
}

<h1 style="color: #FF6347;">Response</h1>


In [33]:
messages = [{'role': 'user', 'content': prompt}]
completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)

In [34]:
answer = completion.choices[0].message.content
print(answer)

# Ethical Dilemmas and Data Challenges in AI

## Critical Perspectives on AI
The discourse surrounding artificial intelligence (AI) is increasingly dominated by ethical considerations, particularly regarding its potential for harmful applications. There is a growing acknowledgment that AI can be used for normatively problematic purposes, which necessitates the establishment of normative principles to guide its development and use (AI for Everyone? Critical Perspectives). Various stakeholders, including governments, corporations, and NGOs, have proposed their own sets of AI ethics principles. However, critiques arise concerning the content of these principles, their implementation, and legal enforceability (Wagner 2018). Moreover, the debate often reflects existing power dynamics where privileged organizations dictate ethical standards without adequately considering marginalized groups’ perspectives or rights.

A significant critical perspective emphasizes the limitations of current eth

<img src="https://miro.medium.com/v2/resize:fit:824/1*GK56xmDIWtNQAD_jnBIt2g.png" alt="NLP Gif" style="width: 500px">

<h2 style="color: #FF6347;">Cosine Similarity</h2>

**Cosine similarity** is a metric used to measure the alignment or similarity between two vectors, calculated as the cosine of the angle between them. It is the **most common metric used in RAG pipelines** for vector retrieval.. It provides a scale from -1 to 1:

- **-1**: Vectors are completely opposite.
- **0**: Vectors are orthogonal (uncorrelated or unrelated).
- **1**: Vectors are identical.


<img src="https://storage.googleapis.com/lds-media/images/cosine-similarity-vectors.original.jpg" alt="NLP Gif" style="width: 700px">

<h2 style="color: #FF6347;">Keyword Highlighting</h2>

Highlighting important keywords helps users quickly understand the relevance of the retrieved text to their query.

In [35]:
from termcolor import colored

The `highlight_keywords` function is designed to highlight specific keywords within a given text. It replaces each keyword in the text with a highlighted version using the `colored` function from the `termcolor` library.


In [36]:
def highlight_keywords(text, keywords):
    for keyword in keywords:
        text = text.replace(keyword, colored(keyword, 'green', attrs=['bold']))
    return text

### Exercice4: add your keywords

In [37]:
query_keywords = ["AI","artificial intelligence",
                  "machine", "human",
                  "social","ethics",
                  "society","ethical",
                  "algorithm","technology"] # add your keywords
for i, doc in enumerate(retrieved_docs[:1]):
    snippet = doc.page_content[:200]
    highlighted = highlight_keywords(snippet, query_keywords)
    print(f"Snippet {i+1}:\n{highlighted}\n{'-'*80}")

Snippet 1:
30 AI for Everyone?
Future research will be needed to address the various challenges with regards to  
the development of artificial intelligence. Which formal method can be used 
to test for algorith
--------------------------------------------------------------------------------


1. `query_keywords` is a list of keywords to be highlighted.
2. The loop iterates over the first document in retrieved_docs.
3. For each document, a snippet of the first 200 characters is extracted.
4. The highlight_keywords function is called to highlight the keywords in the snippet.
5. The highlighted snippet is printed along with a separator line.

<h1 style="color: #FF6347;">Bonus</h1>

**Try loading one of your own PDF books and go through the steps again to explore how the pipeline works with your content**:


# This example uses a PDF from a Python guide, focused on Data Science and Machine Learning  

In [ ]:
# Install the libraries
%pip install langchain langchain_community pypdf
%pip install chromadb langchain_chroma
%pip install langchain_openai openai
%pip install sentence-transformers tiktoken python-dotenv


In [41]:
# Import the libraries
import os
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import openai


# Load my own PDF book 

In [42]:
# Path for PDF book
file_path_python = "../LAB/python_guide.pdf"
# Load the PDF
loader = PyPDFLoader(file_path_python)
pages = loader.load_and_split()

In [44]:
# Create the chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)
chunks = text_splitter.split_documents(pages)

# Embeddings and Vector Store

In [45]:
load_dotenv()
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

db = Chroma.from_documents(
    chunks,
    embeddings,
    persist_directory = "./chroma_db_python"
)

# RAG part

In [47]:
user_question = "¿Qué es una función en Python?"
retrieved_docs = db.similarity_search(user_question, k=5)

# Build the context

In [48]:
def build_context(docs):
    context = ""
    
    for doc in docs: 
        page = doc.metadata.get("page", "N/A")
        context += f"\n(Página {page})\n{doc.page_content}\n"
    return context

formatted_context = build_context(retrieved_docs)

# Prompt control

In [49]:
prompt = f"""
## ROL DEL SISTEMA
Eres un chatbot experto, objetivo y riguroso, diseñado para responder preguntas técnicas sobre **Python**.

Debes basar todas tus respuestas **exclusivamente** en el contenido proporcionado.
No debes inferir, especular ni usar conocimiento externo.

## PREGUNTA DEL USUARIO
"{user_question}"

## CONTEXTO
Fragmentos relevantes del documento:
'''
{formatted_context}
'''

## INSTRUCCIONES
- Usa solo la información del CONTEXTO.
- Si la respuesta no está en el documento, responde:
  "El contexto proporcionado no contiene esta información."
- Usa un tono profesional y claro.
- Cita la página del documento cuando sea posible.

## FORMATO DE RESPUESTA
# [Título breve]
[Respuesta clara y directa]

**Fuente**:
• Guía de Python, Página(s): [...]
"""

print(prompt)


## ROL DEL SISTEMA
Eres un chatbot experto, objetivo y riguroso, diseñado para responder preguntas técnicas sobre **Python**.

Debes basar todas tus respuestas **exclusivamente** en el contenido proporcionado.
No debes inferir, especular ni usar conocimiento externo.

## PREGUNTA DEL USUARIO
"¿Qué es una función en Python?"

## CONTEXTO
Fragmentos relevantes del documento:
'''

(Página 67)
Si una función no devuelve nada, se llama procedimiento: hace su trabajo internamente, pero 
no da un resultado al usuario. 
 
Para definir una función se usa la palabra reservada def, seguida del nombre, paréntesis (para 
los parámetros) y dos puntos. Luego, el bloque de código se escribe con sangría. 
 
 
Aquí tenemos una función llamada sumar_numeros, que recibe dos parámetros (num1 y 
num2), que serán los números que el usuario proporcione. La función simplemente suma esos 
números y devuelve el resultado con return. 
 
Definir la función no la ejecuta. Solo se ejecuta cuando la llamas o invocas

# Use the model

In [51]:
client = openai.OpenAI()

model_params = {
    "model" : "gpt-4o-mini",
    "temperature" : 0.3,
    "max_tokens" : 1500,
    'top_p': 0.9,          
    'frequency_penalty': 0.5,  
    'presence_penalty': 0.6    
}


response = client.chat.completions.create(
    messages = [{"role": "user", "content": prompt}],
    **model_params, 
    timeout=120
)

answer = response.choices[0].message.content
print(answer)

# Definición de una función en Python

Una función en Python es un bloque de código que realiza una tarea específica y puede recibir parámetros. Se define utilizando la palabra reservada `def`, seguida del nombre de la función, paréntesis (para los parámetros) y dos puntos. El bloque de código se escribe con sangría. Si una función no devuelve nada, se denomina procedimiento.

Las funciones se ejecutan cuando son llamadas o invocadas, pasando los valores de los parámetros entre paréntesis. Los parámetros son las variables dentro de la definición de la función, mientras que los argumentos son los valores reales que el usuario proporciona al llamar a la función.

**Fuente**:
• Guía de Python, Página(s): 67


In [ ]:
# The next question, cannot be covered by the PDF, therefore the output should be:
# El contexto proporcionado no contiene esta información. 


In [52]:
user_question = "¿Qué es MLOps?"

prompt = f"""
## ROL DEL SISTEMA
Eres un chatbot experto, objetivo y riguroso, diseñado para responder preguntas técnicas sobre **Python**.

Debes basar todas tus respuestas **exclusivamente** en el contenido proporcionado.
No debes inferir, especular ni usar conocimiento externo.

## PREGUNTA DEL USUARIO
"{user_question}"

## CONTEXTO
Fragmentos relevantes del documento:
'''
{formatted_context}
'''

## INSTRUCCIONES
- Usa solo la información del CONTEXTO.
- Si la respuesta no está en el documento, responde:
  "El contexto proporcionado no contiene esta información."
- Usa un tono profesional y claro.
- Cita la página del documento cuando sea posible.

## FORMATO DE RESPUESTA
# [Título breve]
[Respuesta clara y directa]

**Fuente**:
• Guía de Python, Página(s): [...]
"""


response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [{"role": "user", "content": prompt}],
    temperature = 0.3,
    max_tokens = 1500
)

answer = response.choices[0].message.content
print(answer)



El contexto proporcionado no contiene esta información.
